In [15]:
import pandas as pd

In [16]:
df = pd.read_csv( './ccld-reports-capacities.csv' )

In [17]:
df = pd.read_csv( './ccld-details-full.csv' ).assign( total_enrolled = lambda _df: _df.total_enrolled.fillna( 0 ) ).drop( 'zipcodes', axis = 1 )#.merge( pd.read_csv( '../ccld-reports-capacities.csv' ), on = 'facility_id', how = 'left' )
df

,facility_name,facility_id,facility_comment,recent_capacity,detail_url,report_count,total_enrolled,recent_report_date
0,JCCSF BROTHERHOOD WAY PRESCHOOL,380500086,LICENSED TO SERVE CHILDREN AGED 2-5.5 YRS OLD ...,68,https://www.ccld.dss.ca.gov/transparencyapi/ap...,5,68.0,2022-04-06
1,CALVARY PRESBYTERIAN CHURCH NURSERY SCHOOL,380500099,LICENSED TO SERVE CHILDREN AGES 2 TO ENTRY INT...,75,https://www.ccld.dss.ca.gov/transparencyapi/ap...,3,75.0,2023-05-17
2,GLENRIDGE COOPERATIVE NURSERY SCHOOL,380500240,LICENSED TO SERVE CHILDREN AGED 2.6 - 5.0 YEAR...,45,https://www.ccld.dss.ca.gov/transparencyapi/ap...,6,45.0,2022-01-19
3,HAIGHT ASHBURY COOPERATIVE NURSERY SCHOOL,380500254,"HOURS OF OPERATION: 9:30 AM - 1:00 PM, MONDAY...",24,https://www.ccld.dss.ca.gov/transparencyapi/ap...,4,0.0,2023-02-10
4,HOLY FAMILY DAY HOME,380500308,LICENSED TO SERVE CHILDREN AGES 2 TO ENTRY INT...,150,https://www.ccld.dss.ca.gov/transparencyapi/ap...,9,0.0,2024-01-04
...,...,...,...,...,...,...,...,...
279,POTRERO SPACE PRESCHOOL,384004795,LICENSED TO SERVE 52 CHILDREN AGES TWO YEARS O...,40,https://www.ccld.dss.ca.gov/transparencyapi/ap...,3,40.0,2023-12-12
280,C5 SF PERMIT BUILDING CENTER,384004798,LICENSED TO SERVE 54 CHILDREN AGES TWO YEARS O...,48,https://www.ccld.dss.ca.gov/transparencyapi/ap...,3,48.0,2023-11-29
281,DONUM DEI NEST PRESCHOOL,384004802,LICENSED TO SERVE 12 PRESCHOOL CHILDREN (AGES ...,0,https://www.ccld.dss.ca.gov/transparencyapi/ap...,2,0.0,2024-05-07
282,WONDER YEARS SCHOOL,384004810,LICENSED TO SERVE 45 PRESCHOOL CHILDREN (2 YEA...,0,https://www.ccld.dss.ca.gov/transparencyapi/ap...,3,0.0,2024-11-05


In [18]:
addr_df = pd.read_csv( './data/ccld-sf.csv' )[[ 'facility_id', 'street_address', 'zip_code', 'city' ]]
df = df.merge( addr_df, on = 'facility_id', how = 'left' )
df

,facility_name,facility_id,facility_comment,recent_capacity,detail_url,report_count,total_enrolled,recent_report_date,street_address,zip_code,city
0,JCCSF BROTHERHOOD WAY PRESCHOOL,380500086,LICENSED TO SERVE CHILDREN AGED 2-5.5 YRS OLD ...,68,https://www.ccld.dss.ca.gov/transparencyapi/ap...,5,68.0,2022-04-06,655 BROTHERHOOD WAY,94132.0,SAN FRANCISCO
1,CALVARY PRESBYTERIAN CHURCH NURSERY SCHOOL,380500099,LICENSED TO SERVE CHILDREN AGES 2 TO ENTRY INT...,75,https://www.ccld.dss.ca.gov/transparencyapi/ap...,3,75.0,2023-05-17,2515 FILLMORE STREET,94115.0,SAN FRANCISCO
2,GLENRIDGE COOPERATIVE NURSERY SCHOOL,380500240,LICENSED TO SERVE CHILDREN AGED 2.6 - 5.0 YEAR...,45,https://www.ccld.dss.ca.gov/transparencyapi/ap...,6,45.0,2022-01-19,GLEN PARK CANYON (CHENERY-ELK),94131.0,SAN FRANCISCO
3,HAIGHT ASHBURY COOPERATIVE NURSERY SCHOOL,380500254,"HOURS OF OPERATION: 9:30 AM - 1:00 PM, MONDAY...",24,https://www.ccld.dss.ca.gov/transparencyapi/ap...,4,0.0,2023-02-10,1180 STANYAN STREET,94117.0,SAN FRANCISCO
4,HOLY FAMILY DAY HOME,380500308,LICENSED TO SERVE CHILDREN AGES 2 TO ENTRY INT...,150,https://www.ccld.dss.ca.gov/transparencyapi/ap...,9,0.0,2024-01-04,299 DOLORES STREET,94103.0,SAN FRANCISCO
...,...,...,...,...,...,...,...,...,...,...,...
279,POTRERO SPACE PRESCHOOL,384004795,LICENSED TO SERVE 52 CHILDREN AGES TWO YEARS O...,40,https://www.ccld.dss.ca.gov/transparencyapi/ap...,3,40.0,2023-12-12,824 CAROLINA STREET,94107.0,SAN FRANCISCO
280,C5 SF PERMIT BUILDING CENTER,384004798,LICENSED TO SERVE 54 CHILDREN AGES TWO YEARS O...,48,https://www.ccld.dss.ca.gov/transparencyapi/ap...,3,48.0,2023-11-29,49 SOUTH VAN NESS AVENUE #310,94103.0,SAN FRANCISCO
281,DONUM DEI NEST PRESCHOOL,384004802,LICENSED TO SERVE 12 PRESCHOOL CHILDREN (AGES ...,0,https://www.ccld.dss.ca.gov/transparencyapi/ap...,2,0.0,2024-05-07,1550 TREAT AVENUE,94110.0,SAN FRANCISCO
282,WONDER YEARS SCHOOL,384004810,LICENSED TO SERVE 45 PRESCHOOL CHILDREN (2 YEA...,0,https://www.ccld.dss.ca.gov/transparencyapi/ap...,3,0.0,2024-11-05,380 21ST AVENUE,94121.0,SAN FRANCISCO


In [19]:
import json

with open( './data/ccld-sf-geocache.json' ) as f:
    geocache = json.load( f )

def lookup_latlon( row ):
    if pd.isna( row.street_address ) or pd.isna( row.zip_code ):
        return pd.Series( { 'lat': None, 'lon': None } )
    key = f"{row.street_address}, San Francisco, CA {int(row.zip_code)}"
    coords = geocache.get( key )
    return pd.Series( { 'lat': coords[0] if coords else None, 'lon': coords[1] if coords else None } )

df[[ 'lat', 'lon' ]] = df.apply( lookup_latlon, axis = 1 )
df

,facility_name,facility_id,facility_comment,recent_capacity,detail_url,report_count,total_enrolled,recent_report_date,street_address,zip_code,city,lat,lon
0,JCCSF BROTHERHOOD WAY PRESCHOOL,380500086,LICENSED TO SERVE CHILDREN AGED 2-5.5 YRS OLD ...,68,https://www.ccld.dss.ca.gov/transparencyapi/ap...,5,68.0,2022-04-06,655 BROTHERHOOD WAY,94132.0,SAN FRANCISCO,37.713228,-122.475606
1,CALVARY PRESBYTERIAN CHURCH NURSERY SCHOOL,380500099,LICENSED TO SERVE CHILDREN AGES 2 TO ENTRY INT...,75,https://www.ccld.dss.ca.gov/transparencyapi/ap...,3,75.0,2023-05-17,2515 FILLMORE STREET,94115.0,SAN FRANCISCO,37.792671,-122.434703
2,GLENRIDGE COOPERATIVE NURSERY SCHOOL,380500240,LICENSED TO SERVE CHILDREN AGED 2.6 - 5.0 YEAR...,45,https://www.ccld.dss.ca.gov/transparencyapi/ap...,6,45.0,2022-01-19,GLEN PARK CANYON (CHENERY-ELK),94131.0,SAN FRANCISCO,37.740390,-122.442014
3,HAIGHT ASHBURY COOPERATIVE NURSERY SCHOOL,380500254,"HOURS OF OPERATION: 9:30 AM - 1:00 PM, MONDAY...",24,https://www.ccld.dss.ca.gov/transparencyapi/ap...,4,0.0,2023-02-10,1180 STANYAN STREET,94117.0,SAN FRANCISCO,37.762692,-122.452001
4,HOLY FAMILY DAY HOME,380500308,LICENSED TO SERVE CHILDREN AGES 2 TO ENTRY INT...,150,https://www.ccld.dss.ca.gov/transparencyapi/ap...,9,0.0,2024-01-04,299 DOLORES STREET,94103.0,SAN FRANCISCO,37.764781,-122.426203
...,...,...,...,...,...,...,...,...,...,...,...,...,...
279,POTRERO SPACE PRESCHOOL,384004795,LICENSED TO SERVE 52 CHILDREN AGES TWO YEARS O...,40,https://www.ccld.dss.ca.gov/transparencyapi/ap...,3,40.0,2023-12-12,824 CAROLINA STREET,94107.0,SAN FRANCISCO,37.758242,-122.400249
280,C5 SF PERMIT BUILDING CENTER,384004798,LICENSED TO SERVE 54 CHILDREN AGES TWO YEARS O...,48,https://www.ccld.dss.ca.gov/transparencyapi/ap...,3,48.0,2023-11-29,49 SOUTH VAN NESS AVENUE #310,94103.0,SAN FRANCISCO,37.774494,-122.417791
281,DONUM DEI NEST PRESCHOOL,384004802,LICENSED TO SERVE 12 PRESCHOOL CHILDREN (AGES ...,0,https://www.ccld.dss.ca.gov/transparencyapi/ap...,2,0.0,2024-05-07,1550 TREAT AVENUE,94110.0,SAN FRANCISCO,37.745767,-122.412759
282,WONDER YEARS SCHOOL,384004810,LICENSED TO SERVE 45 PRESCHOOL CHILDREN (2 YEA...,0,https://www.ccld.dss.ca.gov/transparencyapi/ap...,3,0.0,2024-11-05,380 21ST AVENUE,94121.0,SAN FRANCISCO,37.780816,-122.480241


In [20]:
import geopandas as gpd

nhoods = gpd.read_file( 'data/sf-analysis-neighborhoods.geojson' )[[ 'nhood', 'geometry' ]]

pts = gpd.GeoDataFrame( df, geometry = gpd.points_from_xy( df.lon, df.lat ), crs = nhoods.crs )
df = gpd.sjoin( pts, nhoods, how = 'left', predicate = 'within' ).drop( columns = [ 'geometry', 'index_right' ] ).rename( columns = { 'nhood': 'neighborhood' } )
df

,facility_name,facility_id,facility_comment,recent_capacity,detail_url,report_count,total_enrolled,recent_report_date,street_address,zip_code,city,lat,lon,neighborhood
0,JCCSF BROTHERHOOD WAY PRESCHOOL,380500086,LICENSED TO SERVE CHILDREN AGED 2-5.5 YRS OLD ...,68,https://www.ccld.dss.ca.gov/transparencyapi/ap...,5,68.0,2022-04-06,655 BROTHERHOOD WAY,94132.0,SAN FRANCISCO,37.713228,-122.475606,Lakeshore
1,CALVARY PRESBYTERIAN CHURCH NURSERY SCHOOL,380500099,LICENSED TO SERVE CHILDREN AGES 2 TO ENTRY INT...,75,https://www.ccld.dss.ca.gov/transparencyapi/ap...,3,75.0,2023-05-17,2515 FILLMORE STREET,94115.0,SAN FRANCISCO,37.792671,-122.434703,Pacific Heights
2,GLENRIDGE COOPERATIVE NURSERY SCHOOL,380500240,LICENSED TO SERVE CHILDREN AGED 2.6 - 5.0 YEAR...,45,https://www.ccld.dss.ca.gov/transparencyapi/ap...,6,45.0,2022-01-19,GLEN PARK CANYON (CHENERY-ELK),94131.0,SAN FRANCISCO,37.740390,-122.442014,Glen Park
3,HAIGHT ASHBURY COOPERATIVE NURSERY SCHOOL,380500254,"HOURS OF OPERATION: 9:30 AM - 1:00 PM, MONDAY...",24,https://www.ccld.dss.ca.gov/transparencyapi/ap...,4,0.0,2023-02-10,1180 STANYAN STREET,94117.0,SAN FRANCISCO,37.762692,-122.452001,Haight Ashbury
4,HOLY FAMILY DAY HOME,380500308,LICENSED TO SERVE CHILDREN AGES 2 TO ENTRY INT...,150,https://www.ccld.dss.ca.gov/transparencyapi/ap...,9,0.0,2024-01-04,299 DOLORES STREET,94103.0,SAN FRANCISCO,37.764781,-122.426203,Mission
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279,POTRERO SPACE PRESCHOOL,384004795,LICENSED TO SERVE 52 CHILDREN AGES TWO YEARS O...,40,https://www.ccld.dss.ca.gov/transparencyapi/ap...,3,40.0,2023-12-12,824 CAROLINA STREET,94107.0,SAN FRANCISCO,37.758242,-122.400249,Potrero Hill
280,C5 SF PERMIT BUILDING CENTER,384004798,LICENSED TO SERVE 54 CHILDREN AGES TWO YEARS O...,48,https://www.ccld.dss.ca.gov/transparencyapi/ap...,3,48.0,2023-11-29,49 SOUTH VAN NESS AVENUE #310,94103.0,SAN FRANCISCO,37.774494,-122.417791,Mission
281,DONUM DEI NEST PRESCHOOL,384004802,LICENSED TO SERVE 12 PRESCHOOL CHILDREN (AGES ...,0,https://www.ccld.dss.ca.gov/transparencyapi/ap...,2,0.0,2024-05-07,1550 TREAT AVENUE,94110.0,SAN FRANCISCO,37.745767,-122.412759,Bernal Heights
282,WONDER YEARS SCHOOL,384004810,LICENSED TO SERVE 45 PRESCHOOL CHILDREN (2 YEA...,0,https://www.ccld.dss.ca.gov/transparencyapi/ap...,3,0.0,2024-11-05,380 21ST AVENUE,94121.0,SAN FRANCISCO,37.780816,-122.480241,Outer Richmond


In [21]:
df.to_csv( './data/ccld-details-with-neighborhoods.csv', index = False )